In [1]:
%pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.8 MB/s eta 0:00:00


In [16]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2


import keras_tuner as tk

In [3]:
housing = fetch_california_housing(as_frame = True)
df = housing.frame
df


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


In [4]:
df.isna().sum()

,0
MedInc,0
HouseAge,0
AveRooms,0
AveBedrms,0
Population,0
AveOccup,0
Latitude,0
Longitude,0
MedHouseVal,0


In [5]:
X = df.drop('MedHouseVal',axis = 1)
y = df['MedHouseVal']

In [6]:
X_train ,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)




In [8]:
X_test = scaler.transform(X_test)

In [9]:
model = Sequential([Dense(4,activation = "relu",input_shape = (8,)),
                   Dense(1,activation = "linear")])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(optimizer = Adam(learning_rate = 0.001),loss = "mse",metrics = ["mae"])

In [11]:
model.fit(X_train,y_train,epochs = 100,validation_split = 0.2)

Epoch 1/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 2.4674 - mae: 1.1804 - val_loss: 1.3217 - val_mae: 0.8309
Epoch 2/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.9703 - mae: 0.7024 - val_loss: 0.8056 - val_mae: 0.6438
Epoch 3/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.6862 - mae: 0.5922 - val_loss: 0.6436 - val_mae: 0.5754
Epoch 4/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.5610 - mae: 0.5406 - val_loss: 0.5527 - val_mae: 0.5375
Epoch 5/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - loss: 0.4947 - mae: 0.5089 - val_loss: 0.5125 - val_mae: 0.5177
Epoch 6/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.4631 - mae: 0.4922 - val_loss: 0.4919 - val_mae: 0.5015
Epoch 7/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4479 - mae: 0.4819 - val_loss: 0.4797 - val_mae: 0.4986
Epoch 8/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4374 - mae: 0.4751 - val_loss: 0.4708 - val_mae: 0.4934
Epoch 9/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms

In [12]:
test_loss , test_acc = model.evaluate(X_test,y_test)

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3771 - mae: 0.4283


In [13]:
test_acc

0.42834487557411194

In [20]:
def build_model(hp):
  model = Sequential([
      Dense(units = hp.Int("units_1",min_value = 8,max_value = 64 ,step = 8),activation = "relu",input_shape = (8,),kernel_regularizer=l2(0.01)),
      Dense(1,activation = "linear")
  ])

  model.compile(optimizer = Adam(learning_rate = hp.Choice("learning_rate",values=[0.01, 0.001, 0.0001])),loss = "mse",metrics = ["mae"])
  return model

In [21]:
tuner = tk.RandomSearch(
    build_model,
    objective="val_mae",
    max_trials=10,
    executions_per_trial=1,
    directory="ann_tuning",
    project_name="house_price"
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
model1 = tuner.search(X_train, y_train, epochs=10, validation_split=0.2)
model1

Trial 10 Complete [00h 00m 13s]
val_mae: 0.4649542272090912

Best val_mae So Far: 0.45185840129852295
Total elapsed time: 00h 02m 33s


In [23]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
print("R² Score:", r2)

129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
R² Score: 0.7122568195943997


In [25]:
best_model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [26]:
best_model

<Sequential name=sequential, built=True>

In [29]:
y_pred = best_model.evaluate(X_test,y_test)

129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4716 - mae: 0.4452


In [32]:
y_pred_best_model = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred_best_model)
print("R² Score for best model:", r2)

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
R² Score for best model: 0.667075531835005
